In [ ]:
!pip install fastapi uvicorn pyngrok pydantic langgraph groq google-genai faster-whisper soundfile python-dotenv nest_asyncio -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 95.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

folder = "/content/drive/MyDrive/meeting_summarizer/faster_whisper_batch_largev2__MS.ipynb"

print("Folder exists:", os.path.exists(folder))

print("\nFiles inside:")
print(os.listdir("/content/drive/MyDrive/meeting_summarizer/"))

Folder exists: True

Files inside:
['ES2002a.Mix-Headset.wav', 'transcription_results.json', 'all_100_transcripts_with_summary.csv', 'bart_finetuned_final', 'bart_finetuned', 'BART_model_updated.ipynb', 'step6_input_0001.json', 'faster_whisper_batch_largev2__MS.ipynb']


In [ ]:
import os
import sys
import uvicorn
import nest_asyncio
from fastapi import FastAPI, HTTPException, UploadFile, File
from pydantic import BaseModel
from typing import List, Dict, Any
from pyngrok import ngrok
from google.colab import userdata
import shutil

try:
    os.environ["GEMINI2"] = userdata.get("GEMINI2")
    os.environ["NGROK_TOKEN"] = userdata.get("NGROK_TOKEN")
    print("Secrets loaded")
except Exception as e:
    print(f"exception in loading the keys from colab{str(e)}")

%run "/content/drive/MyDrive/meeting_summarizer/faster_whisper_batch_largev2__MS.ipynb"
# %run "/content/drive/MyDrive/meeting_summarizer/bart_summarization.ipynb"

init_shared_models()

# fast api worker on colab
app = FastAPI(title="meeting transcription")

@app.post("/transcribe")
async def transribe_endpoint(file:UploadFile=File(...)):
    try:
      temp_dir="/content/uploaded_audios_MS"
      os.makedirs(temp_dir,exist_ok=True)
      temp_file_path=os.path.join(temp_dir,file.filename)

      with open(temp_file_path,"wb") as buffer:
        shutil.copyfileobj(file.file,buffer)
      print(f"Successfully received file and saved locally to: {temp_file_path}")

      segments = pipeline(temp_file_path)

      if os.path.exists(temp_file_path):
        os.remove(temp_file_path)
      return {
          "transcript": segments

      }
    except Exception as e:
      raise HTTPException(status_code=500, detail=str(e))

def parse_generated_output(raw_output: str):
    marker = "ACTION PLAN:"
    if marker in raw_output:
        summary_part, action_part = raw_output.split(marker, 1)
        return summary_part.replace("SUMMARY:", "", 1).strip(), action_part.strip()
    return raw_output.replace("SUMMARY:", "", 1).strip(), ""

@app.post("/summarize")
async def summarize_endpoint(payload: dict):
    try:
        raw_output = summarize_transcript_v2(payload, mode="match_training")
        summary, action_plan = parse_generated_output(raw_output)
        return {"summary": summary,"action_plan": action_plan, "raw": raw_output}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# RUNTIME GATEWAY---------

ngrok.set_auth_token(os.environ["NGROK_TOKEN"])
ngrok_tunnel = ngrok.connect(8000)
print("Public URL to copy for main.py file:", ngrok_tunnel.public_url)
nest_asyncio.apply()
# Use Config + Server explicitly to stop Uvicorn from creating a fresh conflicting loop
config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)

# Run seamlessly directly inside Colab's active loop
await server.serve()

Secrets loaded
GPU available: True
GPU name     : Tesla T4
VRAM         : 15.6 GB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Logged in as: snehaaiml
Model accessible: pyannote/embedding


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):



 Initializing models...
loading whisper model on CUDA...

loading pyannote pipeline...



config.yaml:   0%|          | 0.00/469 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 5.91MB            

...

pytorch_model.bin: downloading bytes:           |  0.00B            

INFO:     Started server process [1706]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Public URL to copy for main.py file: https://eliminate-silt-liability.ngrok-free.dev
Successfully received file and saved locally to: /content/uploaded_audios_MS/ES2002a.Mix-Headset.wav
  audio duration: 1272.64s

conveting to mono
Original sample rate: 16000Hz
Original shape: (20362240,)
Original dtype: float32
Waveform shape: torch.Size([1, 20362240])
Waveform dtype: torch.float32
 audio prepare (resample)       0.178s

 running transcription...

--- WHISPER SEGMENTS OUTPUT ---
[4.5s - 7.7s]: My gosh, you've already produced a PowerPoint presentation.
[7.8s - 9.3s]: I think it's already on, actually.
[15.1s - 16.2s]: God, I don't know if this is going to work.
[33.2s - 35.0s]: I've plugged it in the back, but...
[39.8s - 40.8s]: OK, right.
[48.0s - 53.8s]: OK. Right.
[56.3s - 59.1s]: Well, this is the kick-off meeting for our project.
[63.6s - 66.6s]: And this is just what we're going to be doing over the next 25 minutes.
[68.5s - 74.4s]: So first of all, just to kind of make sure th

/usr/local/lib/python3.12/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1858.)
  std = sequences.std(dim=-1, correction=1)



PYANNOTE RESULT

[0.0s-0.9s] SPEAKER_03
[4.8s-7.9s] SPEAKER_02
[7.4s-9.5s] SPEAKER_03
[8.6s-9.1s] SPEAKER_02
[10.6s-10.7s] SPEAKER_02
[12.2s-12.7s] SPEAKER_03
[15.0s-16.1s] SPEAKER_03
[19.3s-20.2s] SPEAKER_00
[20.7s-21.6s] SPEAKER_03
[25.9s-27.3s] SPEAKER_01
[30.5s-30.9s] SPEAKER_03
[32.2s-32.9s] SPEAKER_00
[33.4s-35.0s] SPEAKER_03
[35.3s-35.9s] SPEAKER_00
[38.1s-39.1s] SPEAKER_00
[39.8s-42.2s] SPEAKER_03
[42.2s-42.2s] SPEAKER_03
[44.8s-46.1s] SPEAKER_00
[48.2s-49.0s] SPEAKER_00
[48.4s-48.4s] SPEAKER_03
[50.4s-51.1s] SPEAKER_03
[53.6s-54.1s] SPEAKER_03
[55.9s-77.4s] SPEAKER_03
[67.2s-67.3s] SPEAKER_00
[67.3s-67.4s] SPEAKER_02
[67.4s-67.4s] SPEAKER_00
[75.0s-75.1s] SPEAKER_02
[77.4s-80.6s] SPEAKER_00
[80.8s-81.3s] SPEAKER_03
[82.1s-84.5s] SPEAKER_02
[85.9s-88.7s] SPEAKER_01
[89.3s-101.7s] SPEAKER_03
[104.9s-106.2s] SPEAKER_03
[108.7s-132.1s] SPEAKER_03
[132.5s-139.4s] SPEAKER_00
[135.1s-135.5s] SPEAKER_03
[137.1s-137.2s] SPEAKER_02
[139.5s-139.5s] SPEAKER_00
[139.5s-142.0s] SPEAKER_02
